In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.inspection import permutation_importance

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
X, y = make_classification(
    n_samples=1000,
    n_features=5,
    n_informative=3,
    n_redundant=0,
    n_classes=2,
    weights=[0.90, 0.10],
    random_state=42
)

X = pd.DataFrame(
    X,
    columns=[f"Feature_{i}" for i in range(1, 6)]
)

y = pd.Series(y, name="Target")

print("Dataset created successfully!")
print("Dataset shape:", X.shape)
print("\nClass distribution:")
print(y.value_counts())

Dataset created successfully!
Dataset shape: (1000, 5)

Class distribution:
Target
0    894
1    106
Name: count, dtype: int64


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

Training data: (800, 5)
Testing data: (200, 5)

Training class distribution:
Target
0    715
1     85
Name: count, dtype: int64

Testing class distribution:
Target
0    179
1     21
Name: count, dtype: int64


In [5]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Model trained successfully!")
print("Model Accuracy:", accuracy_score(y_test, y_pred))

Model trained successfully!
Model Accuracy: 0.975


In [6]:
results = X_test.copy()

results["Actual"] = y_test
results["Predicted"] = y_pred

results["Error"] = results["Actual"] != results["Predicted"]

print("Total incorrect predictions:", results["Error"].sum())

print("\nIncorrect predictions:")
display(results[results["Error"]])

Total incorrect predictions: 5

Incorrect predictions:


,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Actual,Predicted,Error
635,0.198680,-0.984993,1.027256,0.714189,-1.284611,1,0,True
354,-2.839079,-1.151154,0.345910,0.174666,-0.335325,1,0,True
278,0.570507,2.184097,0.079725,0.708518,-2.424796,1,0,True
601,-0.326696,-0.150320,-1.010696,1.031116,-1.111187,1,0,True
262,0.969569,0.789920,0.416430,0.575434,-1.056076,1,0,True


In [7]:
false_positives = results[
    (results["Actual"] == 0) &
    (results["Predicted"] == 1)
]

false_negatives = results[
    (results["Actual"] == 1) &
    (results["Predicted"] == 0)
]

print("False Positives:", len(false_positives))
print("False Negatives:", len(false_negatives))

print("\nFalse Negative Examples:")
display(false_negatives)

False Positives: 0
False Negatives: 5

False Negative Examples:


,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Actual,Predicted,Error
635,0.198680,-0.984993,1.027256,0.714189,-1.284611,1,0,True
354,-2.839079,-1.151154,0.345910,0.174666,-0.335325,1,0,True
278,0.570507,2.184097,0.079725,0.708518,-2.424796,1,0,True
601,-0.326696,-0.150320,-1.010696,1.031116,-1.111187,1,0,True
262,0.969569,0.789920,0.416430,0.575434,-1.056076,1,0,True


In [8]:
results["Correct"] = results["Actual"] == results["Predicted"]

group_analysis = results.groupby("Actual").agg(
    Total_Records=("Correct", "count"),
    Correct_Predictions=("Correct", "sum"),
    Accuracy=("Correct", "mean")
)

group_analysis["Error_Count"] = (
    group_analysis["Total_Records"] -
    group_analysis["Correct_Predictions"]
)

print("Performance by Actual Class:")
display(group_analysis.round(3))

Performance by Actual Class:


,Total_Records,Correct_Predictions,Accuracy,Error_Count
Actual,,,,
0,179,179,1.000,0
1,21,16,0.762,5


In [9]:
importance = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="f1",
    n_repeats=10,
    random_state=42
)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": importance.importances_mean,
    "Std_Deviation": importance.importances_std
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

display(importance_df.round(4))

,Feature,Importance,Std_Deviation
2,Feature_3,0.5559,0.0468
4,Feature_5,0.3510,0.0540
0,Feature_1,0.0000,0.0000
1,Feature_2,0.0000,0.0000
3,Feature_4,-0.0060,0.0119


In [10]:
error_analysis = false_negatives.copy()

error_analysis["Error_Type"] = "False Negative"
error_analysis["Possible_Reason"] = (
    "Model failed to identify minority class"
)
error_analysis["Next_Experiment"] = (
    "Try class weighting and threshold tuning"
)

error_analysis = error_analysis[
    [
        "Feature_1",
        "Feature_2",
        "Feature_3",
        "Feature_4",
        "Feature_5",
        "Actual",
        "Predicted",
        "Error_Type",
        "Possible_Reason",
        "Next_Experiment"
    ]
]

display(error_analysis)

,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Actual,Predicted,Error_Type,Possible_Reason,Next_Experiment
635,0.198680,-0.984993,1.027256,0.714189,-1.284611,1,0,False Negative,Model failed to identify minority class,Try class weighting and threshold tuning
354,-2.839079,-1.151154,0.345910,0.174666,-0.335325,1,0,False Negative,Model failed to identify minority class,Try class weighting and threshold tuning
278,0.570507,2.184097,0.079725,0.708518,-2.424796,1,0,False Negative,Model failed to identify minority class,Try class weighting and threshold tuning
601,-0.326696,-0.150320,-1.010696,1.031116,-1.111187,1,0,False Negative,Model failed to identify minority class,Try class weighting and threshold tuning
262,0.969569,0.789920,0.416430,0.575434,-1.056076,1,0,False Negative,Model failed to identify minority class,Try class weighting and threshold tuning


In [11]:
# Save error analysis
error_analysis.to_csv("error_analysis.csv", index=False)

# Save feature importance
importance_df.to_csv("feature_importance.csv", index=False)

# Save class-wise performance
group_analysis.to_csv("group_performance.csv")

print("All analysis files saved successfully!")

All analysis files saved successfully!


Proposed Improvements
1. Class Weighting: Use class_weight="balanced" to give more importance to the minority class and improve its recall. Compare precision and F1-score to check the trade-off.
2. Threshold Tuning: Experiment with lower classification thresholds to identify more minority-class records. Select a threshold based on the required balance between precision and recall.